# Telco Customer Churn Prediction

Binary classification pipeline for predicting customer churn in a telecommunications dataset. Five models are trained and evaluated — Random Forest, XGBoost, CatBoost, LightGBM, and Logistic Regression — followed by ensemble methods.

**Primary metric:** F2-score (weights recall twice over precision, appropriate when missing a churner is more costly than a false positive).

In [19]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
    average_precision_score, precision_recall_curve,
    auc, fbeta_score, make_scorer
)
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import optuna
import shap
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [20]:
df = pd.read_csv('../datasets/Cópia de WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [21]:
print('DATA QUALITY REPORT')

numeric_feature_cols = df.select_dtypes(include=['number']).columns.tolist()

#Missing values
missing_total   = int(df.isnull().sum().sum())
missing_by_col = df.isnull().sum()
print('')
print('Missing Values:', missing_total, 'total')
if missing_total > 0:
    print(missing_by_col[missing_by_col > 0].to_string())
else:
    print('  None')

#Duplicate records
duplicate_rows = int(df.duplicated().sum())
print('')
print('Duplicate Records: {} ({:.4f}%)'.format(duplicate_rows, duplicate_rows / len(df) * 100))

#Outliers - IQR method
q1  = df[numeric_feature_cols].quantile(0.25)
q3  = df[numeric_feature_cols].quantile(0.75)
iqr = q3 - q1
outlier_mask = (
    (df[numeric_feature_cols] < (q1 - 1.5 * iqr)) |
    (df[numeric_feature_cols] > (q3 + 1.5 * iqr))
)
outlier_rows = outlier_mask.any(axis=1).sum()
print('')
print('Rows with at least one outlier (IQR): {} ({:.2f}%)'.format(
    outlier_rows, outlier_rows / len(df) * 100))

lower_bound = (q1 - 1.5 * iqr).rename('Lower Bound')
upper_bound = (q3 + 1.5 * iqr).rename('Upper Bound')
outlier_detail = pd.DataFrame({
    'Lower Bound': lower_bound,
    'Upper Bound': upper_bound,
    'N Outliers':  outlier_mask.sum(),
    '% Outliers':  (outlier_mask.sum() / len(df) * 100).round(2),
    'Min Outlier': [df.loc[outlier_mask[c], c].min() if outlier_mask[c].any() else float('nan') for c in numeric_feature_cols],
    'Max Outlier': [df.loc[outlier_mask[c], c].max() if outlier_mask[c].any() else float('nan') for c in numeric_feature_cols],
}).rename_axis('Feature')
outlier_detail = (
    outlier_detail[outlier_detail['N Outliers'] > 0]
    .sort_values('N Outliers', ascending=False)
)
print('')
display(outlier_detail)

del outlier_mask, lower_bound, upper_bound, outlier_detail
del numeric_feature_cols, missing_total, missing_by_col, duplicate_rows, q1, q3, iqr, outlier_rows

DATA QUALITY REPORT

Missing Values: 0 total
  None

Duplicate Records: 0 (0.0000%)

Rows with at least one outlier (IQR): 1142 (16.21%)



,Lower Bound,Upper Bound,N Outliers,% Outliers,Min Outlier,Max Outlier
Feature,,,,,,
SeniorCitizen,0.0,0.0,1142,16.21,1.0,1.0


In [22]:
#Initial inspection
pd.set_option('display.max_columns', None)
print(f'Dataset shape: {df.shape}')
df.info()

#Drop non-informative identifier
df = df.drop(['customerID'], axis=1)

#Fix TotalCharges: coerce empty strings to NaN (11 rows with tenure = 0)
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')

#Remove rows where tenure = 0 (no usage history; TotalCharges is NaN)
zero_tenure_idx = df[df['tenure'] == 0].index
df.drop(labels=zero_tenure_idx, axis=0, inplace=True)

assert df.isnull().sum().sum() == 0, 'Unexpected nulls remain after cleaning.'
print(f'Cleaned dataset shape: {df.shape}')


Dataset shape: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling

## 1. Feature Engineering

In [23]:
#Customer lifetime value proxies
df['CLV_proxy'] = df['TotalCharges'] / (df['tenure'] + 1)
df['Avg_monthly_spend'] = df['TotalCharges'] / df['tenure']

#Service bundle count
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_services'] = df[service_cols].apply(lambda x: sum([1 for val in x if val == 'Yes']), axis=1)

#Log-transform skewed charge features
df['MonthlyCharges_log'] = np.log1p(df['MonthlyCharges'])
df['TotalCharges_log'] = np.log1p(df['TotalCharges'])

In [24]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

display(df.head())
df[numerical_cols].describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CLV_proxy,Avg_monthly_spend,Total_services,MonthlyCharges_log,TotalCharges_log
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,14.925000,29.850000,1,3.429137,3.429137
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,53.985714,55.573529,2,4.059581,7.544597
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,36.050000,54.075000,2,4.004602,4.692723
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,40.016304,40.905556,3,3.768153,7.518471
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,50.550000,75.825000,0,4.272491,5.028148


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,CLV_proxy,Avg_monthly_spend,Total_services,MonthlyCharges_log,TotalCharges_log
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,32.421786,64.798208,2283.300441,59.083067,64.799424,2.038111,4.044039,6.943387
std,0.368844,24.545260,30.085974,2266.771362,30.514438,30.185891,1.847161,0.580080,1.546438
min,0.000000,1.000000,18.250000,18.800000,9.183333,13.775000,0.000000,2.957511,2.985682
25%,0.000000,9.000000,35.587500,401.450000,26.225944,36.179891,0.000000,3.599706,5.997571
50%,0.000000,29.000000,70.350000,1397.475000,61.070387,70.373239,2.000000,4.267597,7.243138
75%,0.000000,55.000000,89.862500,3794.737500,84.877538,90.179560,3.000000,4.509347,8.241634
max,1.000000,72.000000,118.750000,8684.800000,118.969863,121.400000,6.000000,4.785406,9.069445


In [25]:
#Get the raw counts (How many 'Yes' vs 'No')
churn_counts = df['Churn'].value_counts()


churn_percentages = df['Churn'].value_counts(normalize=True) * 100

#Combine into a readable summary
churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage (%)': churn_percentages
})

print("--- Churn Overview ---")
print(churn_summary)

--- Churn Overview ---
       Count  Percentage (%)
Churn                       
No      5163       73.421502
Yes     1869       26.578498


## 2. Preprocessing Pipelines

Four pipelines are constructed to satisfy the input requirements of each model family:

| Pipeline | Models | Encoding | Scaling |
|---|---|---|---|
| 1 | CatBoost | Raw categoricals | None |
| 2 | LightGBM | `category` dtype | None |
| 3 | XGBoost, Random Forest | One-hot | None |
| 4 | Logistic Regression | One-hot | StandardScaler (continuous only) |

One-hot encoding is fitted on the training split only; test columns are aligned via `reindex`.

In [26]:
#Preserve the original feature representation for CatBoost.
df_original = df.copy()

#Convert target variable for original df
df_original['Churn'] = df_original['Churn'].map({'No': 0, 'Yes': 1})

#CatBoost pipeline (categorical features as-is, no standardization)
categorical_cols_original = df_original.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in categorical_cols_original:
    categorical_cols_original.remove('Churn')

#Get categorical column indices for CatBoost
cat_features_indices = [df_original.drop('Churn', axis=1).columns.get_loc(col) 
                        for col in categorical_cols_original]

X_original = df_original.drop('Churn', axis=1)
y_original = df_original['Churn']

X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42, stratify=y_original
)

#LightGBM pipeline (category dtype, no standardization)
df_label = df.copy()
df_label['Churn'] = df_label['Churn'].map({'No': 0, 'Yes': 1})

#Cast categorical columns to 'category' dtype
categorical_cols_label = df_label.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in categorical_cols_label:
    categorical_cols_label.remove('Churn')

for col in categorical_cols_label:
    df_label[col] = df_label[col].astype('category')

#Mark categorical features for LightGBM
categorical_feature_names = categorical_cols_label

X_label = df_label.drop('Churn', axis=1)
y_label = df_label['Churn']

X_train_label, X_test_label, y_train_label, y_test_label = train_test_split(
    X_label, y_label, test_size=0.2, random_state=42, stratify=y_label
)

#XGBoost and Random Forest pipeline (one-hot encoding, no standardization)
df_onehot = df.copy()

#Apply one-hot encoding
categorical_cols_onehot = df_onehot.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in categorical_cols_onehot:
    categorical_cols_onehot.remove('Churn')

#Convert target variable
df_onehot['Churn'] = df_onehot['Churn'].map({'No': 0, 'Yes': 1})

#Define features and target
X_onehot_raw = df_onehot.drop('Churn', axis=1)
y_onehot     = df_onehot['Churn']

#Split BEFORE encoding to prevent data leakage
X_train_onehot_raw, X_test_onehot_raw, y_train_onehot, y_test_onehot = train_test_split(
    X_onehot_raw, y_onehot, test_size=0.2, random_state=42, stratify=y_onehot
)

#Fit encoding on train only, then align test columns to train
X_train_onehot = pd.get_dummies(X_train_onehot_raw, columns=categorical_cols_onehot, drop_first=True)
X_test_onehot  = pd.get_dummies(X_test_onehot_raw,  columns=categorical_cols_onehot, drop_first=True)
X_test_onehot  = X_test_onehot.reindex(columns=X_train_onehot.columns, fill_value=0)

bool_cols = X_train_onehot.select_dtypes(include=['bool']).columns
X_train_onehot[bool_cols] = X_train_onehot[bool_cols].astype(int)
X_test_onehot[bool_cols]  = X_test_onehot[bool_cols].astype(int)


#Logistic Regression pipeline (one-hot encoding and standardization)

df_onehot_scaled = df.copy()

categorical_cols_scaled = df_onehot_scaled.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in categorical_cols_scaled:
    categorical_cols_scaled.remove('Churn')

#Convert target variable
df_onehot_scaled['Churn'] = df_onehot_scaled['Churn'].map({'No': 0, 'Yes': 1})

#Define features and target
X_scaled_raw = df_onehot_scaled.drop('Churn', axis=1)
y_scaled_all = df_onehot_scaled['Churn']

#Split BEFORE encoding to prevent data leakage
X_train_scaled_raw, X_test_scaled_raw, y_train_scaled, y_test_scaled = train_test_split(
    X_scaled_raw, y_scaled_all, test_size=0.2, random_state=42, stratify=y_scaled_all
)

#Fit encoding on train only, then align test columns to train
X_train_scaled_base = pd.get_dummies(X_train_scaled_raw, columns=categorical_cols_scaled, drop_first=True)
X_test_scaled_base  = pd.get_dummies(X_test_scaled_raw,  columns=categorical_cols_scaled, drop_first=True)

X_test_scaled_base  = X_test_scaled_base.reindex(columns=X_train_scaled_base.columns, fill_value=0)

#Convert boolean columns to integers
bool_cols_scaled = X_train_scaled_base.select_dtypes(include=['bool']).columns
X_train_scaled_base[bool_cols_scaled] = X_train_scaled_base[bool_cols_scaled].astype(int)
X_test_scaled_base[bool_cols_scaled]  = X_test_scaled_base[bool_cols_scaled].astype(int)

#Standardization (automatic detection of continuous features)

continuous_cols = [
    col for col in X_train_scaled_base.select_dtypes(include=['int64', 'float64']).columns
    if X_train_scaled_base[col].nunique() > 2
]

cols_to_scale = [c for c in continuous_cols if c in X_train_scaled_base.columns]

X_train_scaled = X_train_scaled_base.copy()
X_test_scaled  = X_test_scaled_base.copy()

scaler = StandardScaler()

if len(cols_to_scale) > 0:
    X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_scaled_base[cols_to_scale])
    X_test_scaled[cols_to_scale]  = scaler.transform(X_test_scaled_base[cols_to_scale])

#Drop raw versions of log-transformed features (avoid multicollinearity)
#Only affects Pipeline 4 (Logistic Regression) - other pipelines untouched
cols_to_drop_for_logreg = ['MonthlyCharges', 'TotalCharges']
cols_to_drop_for_logreg = [c for c in cols_to_drop_for_logreg if c in X_train_scaled.columns]
X_train_scaled = X_train_scaled.drop(columns=cols_to_drop_for_logreg)
X_test_scaled  = X_test_scaled.drop(columns=cols_to_drop_for_logreg)

print(f"Dropped {cols_to_drop_for_logreg} from LogReg pipeline (log versions kept)")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

Dropped ['MonthlyCharges', 'TotalCharges'] from LogReg pipeline (log versions kept)
X_train_scaled shape: (5625, 33)
X_test_scaled shape:  (1407, 33)


## 3. Hyperparameter Tuning

Optuna (TPE sampler, 80 trials per model) optimizes the F2-score through stratified 5-fold cross-validation. Class imbalance is handled per model:

| Model | Imbalance strategy |
|---|---|
| Random Forest | SMOTEENN inside each CV fold |
| XGBoost | `scale_pos_weight` |
| CatBoost | `auto_class_weights='Balanced'` |
| LightGBM | `is_unbalance=True` |
| Logistic Regression | SMOTEENN inside each CV fold |

In [27]:
f2_scorer = make_scorer(fbeta_score, beta=2)

#Random Forest with SMOTEENN (inside each CV fold)
#Hyperparameter search notes
#max_depth search range set to 2-15
#n_estimators search range set to 100-1000
#min_samples_leaf search range set to 1-30
#min_samples_split upper bound set to 30
print("1. TUNING RANDOM FOREST (with SMOTEENN inside CV folds)")

def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'max_depth':         trial.suggest_int('max_depth', 2, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 30),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap':         trial.suggest_categorical('bootstrap', [True, False]),
        'random_state':      SEED,
        'n_jobs':            -1
    }

    model = RandomForestClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train_onehot, y_train_onehot):
        X_train_fold = X_train_onehot.iloc[train_idx]
        X_val_fold   = X_train_onehot.iloc[val_idx]
        y_train_fold = y_train_onehot.iloc[train_idx]
        y_val_fold   = y_train_onehot.iloc[val_idx]

        smote_enn = SMOTEENN(random_state=SEED)
        X_resampled, y_resampled = smote_enn.fit_resample(X_train_fold, y_train_fold)

        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_val_fold)
        scores.append(fbeta_score(y_val_fold, y_pred, beta=2))

    return np.mean(scores)

study_rf = optuna.create_study(direction='maximize', study_name='RandomForest',
                                sampler=TPESampler(seed=SEED))
study_rf.optimize(objective_rf, n_trials=80, show_progress_bar=True)
print(f"\nBest Random Forest F2 Score: {study_rf.best_value:.4f}")
print("Best Random Forest parameters:", study_rf.best_params)

#XGBoost with ClassWeight
#Hyperparameter search notes
#min_child_weight search range set to 1-10
#reg_alpha search range set to 0-10
#reg_lambda search range set to 0-10
#max_depth search range kept at 2-12
print("2. TUNING XGBOOST (with ClassWeight)")

scale_pos_xgb = len(y_train_onehot[y_train_onehot == 0]) / len(y_train_onehot[y_train_onehot == 1])

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 2, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0, 10),
        'scale_pos_weight':  scale_pos_xgb,
        'eval_metric':       'logloss',
        'random_state':      SEED,
        'n_jobs':            -1
    }

    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    score = cross_val_score(model, X_train_onehot, y_train_onehot,
                            cv=cv, scoring=f2_scorer).mean()
    return score

study_xgb = optuna.create_study(direction='maximize', study_name='XGBoost',
                                 sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=80, show_progress_bar=True)
print(f"\nBest XGBoost F2 Score: {study_xgb.best_value:.4f}")
print("Best XGBoost parameters:", study_xgb.best_params)

#CatBoost with ClassWeight
#Hyperparameter search notes
#rsm search range set to 0.5-1.0
#bagging_temperature search range set to 0-1
#min_data_in_leaf search range set to 1-50
#CatBoost uses random_seed
print("3. TUNING CATBOOST (with ClassWeight)")

def objective_catboost(trial):
    params = {
        'iterations':           trial.suggest_int('iterations', 100, 500),
        'depth':                trial.suggest_int('depth', 2, 10),
        'learning_rate':        trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg':          trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count':         trial.suggest_int('border_count', 32, 255),
        'rsm':                  trial.suggest_float('rsm', 0.5, 1.0),
        'bagging_temperature':  trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'min_data_in_leaf':     trial.suggest_int('min_data_in_leaf', 1, 50),
        'auto_class_weights':   'Balanced',
        'random_seed':          SEED,
        'verbose':              False
    }

    model = CatBoostClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train_orig, y_train_orig):
        X_train_fold = X_train_orig.iloc[train_idx]
        X_val_fold   = X_train_orig.iloc[val_idx]
        y_train_fold = y_train_orig.iloc[train_idx]
        y_val_fold   = y_train_orig.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold,
                  cat_features=cat_features_indices, verbose=False)
        y_pred = model.predict(X_val_fold)
        scores.append(fbeta_score(y_val_fold, y_pred, beta=2))

    return np.mean(scores)

study_catboost = optuna.create_study(direction='maximize', study_name='CatBoost',
                                      sampler=TPESampler(seed=SEED))
study_catboost.optimize(objective_catboost, n_trials=80, show_progress_bar=True)
print(f"\nBest CatBoost F2 Score: {study_catboost.best_value:.4f}")
print("Best CatBoost parameters:", study_catboost.best_params)

#LightGBM with ClassWeight
#Hyperparameter search notes
#num_leaves search range set to 20-300
#min_child_samples search range set to 10-100
#max_depth search range kept at 2-12
print("4. TUNING LIGHTGBM (with ClassWeight)")

def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 2, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0, 10),
        'is_unbalance':      True,
        'random_state':      SEED,
        'n_jobs':            1,
        'verbose':           -1
    }

    model = LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train_label, y_train_label):
        X_train_fold = X_train_label.iloc[train_idx]
        X_val_fold   = X_train_label.iloc[val_idx]
        y_train_fold = y_train_label.iloc[train_idx]
        y_val_fold   = y_train_label.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold, categorical_feature='auto')
        y_pred = model.predict(X_val_fold)
        scores.append(fbeta_score(y_val_fold, y_pred, beta=2))

    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize', study_name='LightGBM',
                                  sampler=TPESampler(seed=SEED))
study_lgbm.optimize(objective_lgbm, n_trials=40, show_progress_bar=True)
print(f"\nBest LightGBM F2 Score: {study_lgbm.best_value:.4f}")
print("Best LightGBM parameters:", study_lgbm.best_params)

#Logistic Regression with SMOTEENN (inside each CV fold)
#Hyperparameter search notes
#max_iter fixed outside the search space
#Convergence setting fixed at 2000
print("5. TUNING LOGISTIC REGRESSION (with SMOTEENN inside CV folds)")

def objective_logreg(trial):
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])

    params = {
        'C':            trial.suggest_float('C', 0.001, 100, log=True),
        'penalty':      penalty,
        'solver':       'saga',
        'max_iter':     2000,       #Fixed convergence setting
        'random_state': SEED,
        'n_jobs':       -1
    }

    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0, 1)

    model = LogisticRegression(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train_scaled, y_train_scaled):
        X_train_fold = X_train_scaled.iloc[train_idx]
        X_val_fold   = X_train_scaled.iloc[val_idx]
        y_train_fold = y_train_scaled.iloc[train_idx]
        y_val_fold   = y_train_scaled.iloc[val_idx]

        smote_enn = SMOTEENN(random_state=SEED)
        X_resampled, y_resampled = smote_enn.fit_resample(X_train_fold, y_train_fold)

        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_val_fold)
        scores.append(fbeta_score(y_val_fold, y_pred, beta=2))

    return np.mean(scores)

study_logreg = optuna.create_study(direction='maximize', study_name='LogisticRegression',
                                    sampler=TPESampler(seed=SEED))
study_logreg.optimize(objective_logreg, n_trials=80, show_progress_bar=True)
print(f"\nBest Logistic Regression F2 Score: {study_logreg.best_value:.4f}")
print("Best Logistic Regression parameters:", study_logreg.best_params)

#Summary
print("SUMMARY: BEST HYPERPARAMETERS FOR EACH MODEL")

results_summary = {
    'RandomForest (SMOTEENN)':       {'F2_Score': study_rf.best_value,      'Parameters': study_rf.best_params},
    'XGBoost (ClassWeight)':         {'F2_Score': study_xgb.best_value,     'Parameters': study_xgb.best_params},
    'CatBoost (ClassWeight)':        {'F2_Score': study_catboost.best_value, 'Parameters': study_catboost.best_params},
    'LightGBM (ClassWeight)':        {'F2_Score': study_lgbm.best_value,     'Parameters': study_lgbm.best_params},
    'LogisticRegression (SMOTEENN)': {'F2_Score': study_logreg.best_value,   'Parameters': study_logreg.best_params},
}

for model_name, results in results_summary.items():
    print(f"\n{model_name}:")
    print(f"  F2 Score: {results['F2_Score']:.4f}")
    print(f"  Best Parameters: {results['Parameters']}")



1. TUNING RANDOM FOREST (with SMOTEENN inside CV folds)


Best trial: 76. Best value: 0.719254: 100%|██████████| 80/80 [12:04<00:00,  9.05s/it]



Best Random Forest F2 Score: 0.7193
Best Random Forest parameters: {'n_estimators': 1000, 'max_depth': 2, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}
2. TUNING XGBOOST (with ClassWeight)


Best trial: 68. Best value: 0.737335: 100%|██████████| 80/80 [8:51:51<00:00, 398.90s/it]   



Best XGBoost F2 Score: 0.7373
Best XGBoost parameters: {'n_estimators': 206, 'max_depth': 3, 'learning_rate': 0.018558023949286063, 'subsample': 0.70048901508101, 'colsample_bytree': 0.8356843348234179, 'gamma': 3.4795846079913724, 'min_child_weight': 8, 'reg_alpha': 5.4980779862484574, 'reg_lambda': 0.9519265508655455}
3. TUNING CATBOOST (with ClassWeight)


...


Best CatBoost F2 Score: 0.7359
Best CatBoost parameters: {'iterations': 271, 'depth': 3, 'learning_rate': 0.034000431380307325, 'l2_leaf_reg': 6.602089419019067, 'border_count': 178, 'rsm': 0.6835819698970476, 'bagging_temperature': 0.042745180079641565, 'min_data_in_leaf': 45}
4. TUNING LIGHTGBM (with ClassWeight)


Best trial: 16. Best value: 0.731972: 100%|██████████| 40/40 [00:38<00:00,  1.04it/s]



Best LightGBM F2 Score: 0.7320
Best LightGBM parameters: {'n_estimators': 165, 'max_depth': 2, 'learning_rate': 0.0344462350715294, 'num_leaves': 90, 'min_child_samples': 47, 'subsample': 0.7751231143623358, 'colsample_bytree': 0.9018036373320125, 'reg_alpha': 4.582181452313923, 'reg_lambda': 3.3848344546818883}
5. TUNING LOGISTIC REGRESSION (with SMOTEENN inside CV folds)


Best trial: 0. Best value: 0.739627:   1%|▏         | 1/80 [00:05<06:46,  5.15s/it]The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
Best trial: 30. Best value: 0.747469:  46%|████▋     | 37/80 [03:53<03:18,  4.62s/it]The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the coef_ did not converge
Best trial: 30. Best value: 0.747469:  74%|███████▍  | 59/80 [06:07<01:45,  5.04s/it]The max_iter was reached which means the coef_ did not converge
The max_iter was reached which means the co


Best Logistic Regression F2 Score: 0.7475
Best Logistic Regression parameters: {'penalty': 'l1', 'C': 0.040057099563026936}
SUMMARY: BEST HYPERPARAMETERS FOR EACH MODEL

RandomForest (SMOTEENN):
  F2 Score: 0.7193
  Best Parameters: {'n_estimators': 1000, 'max_depth': 2, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}

XGBoost (ClassWeight):
  F2 Score: 0.7373
  Best Parameters: {'n_estimators': 206, 'max_depth': 3, 'learning_rate': 0.018558023949286063, 'subsample': 0.70048901508101, 'colsample_bytree': 0.8356843348234179, 'gamma': 3.4795846079913724, 'min_child_weight': 8, 'reg_alpha': 5.4980779862484574, 'reg_lambda': 0.9519265508655455}

CatBoost (ClassWeight):
  F2 Score: 0.7359
  Best Parameters: {'iterations': 271, 'depth': 3, 'learning_rate': 0.034000431380307325, 'l2_leaf_reg': 6.602089419019067, 'border_count': 178, 'rsm': 0.6835819698970476, 'bagging_temperature': 0.042745180079641565, 'min_data_in_leaf': 45}

LightGBM (ClassWeight

## 4. Model Training and Threshold Optimization

Models are trained on the full training set using the best Optuna parameters. Classification thresholds are optimized using out-of-fold (OOF) predictions on the training set only — the test set is never consulted during threshold selection.

In [28]:
def get_oof_probabilities(model_type, X_train_data, y_train_data,
                          model_params, n_splits=5):
    """
    Generate out-of-fold probability predictions on the training set.
    SMOTEENN is applied inside each fold for RF and LogReg.
    CatBoost and LightGBM use their built-in class balancing.

    Parameters
    model_type : str
        One of 'rf', 'xgb', 'catboost', 'lgbm', 'logreg'
    X_train_data : pd.DataFrame
        Raw (non-resampled) training features in the correct format for
        the given model type.
    y_train_data : pd.Series
        Training labels.
    model_params : dict
        Best hyperparameters from Optuna (will not be mutated).
    n_splits : int
        Number of CV folds.

    Returns
    -------
    oof_proba : np.ndarray, shape (n_train_samples,)
        OOF predicted probabilities for the positive class.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_proba = np.zeros(len(X_train_data))
    params = model_params.copy()

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_data, y_train_data)):
        X_fold_tr = X_train_data.iloc[train_idx]
        X_fold_val = X_train_data.iloc[val_idx]
        y_fold_tr = y_train_data.iloc[train_idx]

        if model_type == 'rf':
            smote_enn = SMOTEENN(random_state=42)
            X_fold_tr, y_fold_tr = smote_enn.fit_resample(X_fold_tr, y_fold_tr)
            fold_model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
            fold_model.fit(X_fold_tr, y_fold_tr)

        elif model_type == 'xgb':
            params.update({
                'scale_pos_weight': scale_pos_xgb,
                'eval_metric': 'logloss',
                'random_state': 42,
                'n_jobs': -1
            })
            fold_model = XGBClassifier(**params)
            fold_model.fit(X_fold_tr, y_fold_tr)

        elif model_type == 'catboost':
            params.update({
                'auto_class_weights': 'Balanced',
                'random_seed': 42,
                'verbose': False
            })
            fold_model = CatBoostClassifier(**params)
            fold_model.fit(X_fold_tr, y_fold_tr,
                           cat_features=cat_features_indices, verbose=False)

        elif model_type == 'lgbm':
            params.update({
                'is_unbalance': True,
                'random_state': 42,
                'n_jobs': 1,
                'verbose': -1
            })
            fold_model = LGBMClassifier(**params)
            fold_model.fit(X_fold_tr, y_fold_tr, categorical_feature='auto')

        elif model_type == 'logreg':
            params.update({
                'solver': 'saga',
                'max_iter': 2000,
                'random_state': 42,
                'n_jobs': -1
            })
            smote_enn = SMOTEENN(random_state=42)
            X_fold_tr, y_fold_tr = smote_enn.fit_resample(X_fold_tr, y_fold_tr)
            fold_model = LogisticRegression(**params)
            fold_model.fit(X_fold_tr, y_fold_tr)

        oof_proba[val_idx] = fold_model.predict_proba(X_fold_val)[:, 1]

    return oof_proba


def find_optimal_threshold_oof(oof_proba, y_true):
    """
    Sweep thresholds on OOF predictions and return the one that
    maximises the F2 score.

    Parameters
    oof_proba : np.ndarray
        OOF predicted probabilities (training set only).
    y_true : pd.Series or np.ndarray
        True training labels (same ordering as oof_proba).

    Returns
    -------
    best_threshold : float
    best_f2 : float
    """
    thresholds = np.arange(0.10, 0.90, 0.01)
    best_threshold, best_f2 = 0.5, 0.0

    for t in thresholds:
        y_pred = (oof_proba >= t).astype(int)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f2   = (5 * prec * rec) / (4 * prec + rec) if (prec + rec) > 0 else 0.0
        if f2 > best_f2:
            best_f2, best_threshold = f2, t

    return best_threshold, best_f2

In [29]:
#Create resampled datasets
print("Step 0: Creating resampled datasets")

smote_enn_rf = SMOTEENN(random_state=42)
X_train_rf_resampled, y_train_rf_resampled = smote_enn_rf.fit_resample(
    X_train_onehot, y_train_onehot
)
print(f"Random Forest: {X_train_onehot.shape}  to  {X_train_rf_resampled.shape}")

smote_enn_lr = SMOTEENN(random_state=42)
X_train_lr_resampled, y_train_lr_resampled = smote_enn_lr.fit_resample(
    X_train_scaled, y_train_scaled
)
print(f"LogReg: {X_train_scaled.shape}  to  {X_train_lr_resampled.shape}")

scale_pos_xgb = len(y_train_onehot[y_train_onehot == 0]) / len(y_train_onehot[y_train_onehot == 1])
print(f"XGBoost scale_pos_weight: {scale_pos_xgb:.3f}")

#Retrieve Optuna hyperparameters
print("Step 1: Retrieving Optuna hyperparameters")

rf_best_params       = study_rf.best_params.copy()
xgb_best_params      = study_xgb.best_params.copy()
catboost_best_params = study_catboost.best_params.copy()
lgbm_best_params     = study_lgbm.best_params.copy()
logreg_best_params   = study_logreg.best_params.copy()

xgb_params_final = xgb_best_params.copy()
xgb_params_final.update({
    'scale_pos_weight': scale_pos_xgb,
    'eval_metric': 'logloss',
    'random_state': SEED,
    'n_jobs': -1
})

catboost_params_final = catboost_best_params.copy()
catboost_params_final.update({
    'auto_class_weights': 'Balanced',
    'random_seed': SEED,
    'verbose': False
})

lgbm_params_final = lgbm_best_params.copy()
lgbm_params_final.update({
    'is_unbalance': True,
    'random_state': SEED,
    'n_jobs': 1,
    'verbose': -1
})

logreg_params_final = logreg_best_params.copy()
logreg_params_final.update({
    'solver': 'saga',
    'max_iter': 2000,
    'random_state': SEED,
    'n_jobs': -1
})

print("Best parameters retrieved and fixed training parameters restored for all 5 models\n")

#Train models on the full training data
print("Step 2: Training models with selected hyperparameters")

print("Training Random Forest with SMOTEENN")
rf_model = RandomForestClassifier(**rf_best_params, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf_resampled, y_train_rf_resampled)
print("Random Forest trained")

print("Training XGBoost with ClassWeight")
xgb_model = XGBClassifier(**xgb_params_final)
xgb_model.fit(X_train_onehot, y_train_onehot)
print("XGBoost trained")

print("Training CatBoost with ClassWeight")
catboost_model = CatBoostClassifier(**catboost_params_final)
catboost_model.fit(X_train_orig, y_train_orig, cat_features=cat_features_indices)
print("CatBoost trained")

print("Training LightGBM with ClassWeight (is_unbalance=True)")
lgbm_model = LGBMClassifier(**lgbm_params_final)
lgbm_model.fit(X_train_label, y_train_label, categorical_feature='auto')
print("LightGBM trained")

print("Training Logistic Regression with SMOTEENN")
logreg_model = LogisticRegression(**logreg_params_final)
logreg_model.fit(X_train_lr_resampled, y_train_lr_resampled)
print("Logistic Regression trained")

print("\nAll 5 models trained successfully!\n")

#Generate holdout probabilities for final evaluation
print("Step 3: Generating holdout probability predictions")

y_pred_proba_rf       = rf_model.predict_proba(X_test_onehot)[:, 1]
y_pred_proba_xgb      = xgb_model.predict_proba(X_test_onehot)[:, 1]
y_pred_proba_catboost = catboost_model.predict_proba(X_test_orig)[:, 1]
y_pred_proba_lgbm     = lgbm_model.predict_proba(X_test_label)[:, 1]
y_pred_proba_logreg   = logreg_model.predict_proba(X_test_scaled)[:, 1]

print("Test set probability predictions generated for all models\n")

Step 0: Creating resampled datasets
Random Forest: (5625, 35)  to  (4687, 35)
LogReg: (5625, 33)  to  (5409, 33)
XGBoost scale_pos_weight: 2.763
Step 1: Retrieving Optuna hyperparameters
Best parameters retrieved and fixed training parameters restored for all 5 models

Step 2: Training models with selected hyperparameters
Training Random Forest with SMOTEENN
Random Forest trained
Training XGBoost with ClassWeight
XGBoost trained
Training CatBoost with ClassWeight
CatBoost trained
Training LightGBM with ClassWeight (is_unbalance=True)
LightGBM trained
Training Logistic Regression with SMOTEENN
Logistic Regression trained

All 5 models trained successfully!

Step 3: Generating holdout probability predictions
Test set probability predictions generated for all models



In [30]:
#Generate OOF probabilities for each model on its native training data
print("Generating OOF probabilities (this runs 5-fold CV per model)")

print("  Random Forest")
oof_rf = get_oof_probabilities('rf', X_train_onehot, y_train_onehot, rf_best_params)

print("  XGBoost")
oof_xgb = get_oof_probabilities('xgb', X_train_onehot, y_train_onehot, xgb_best_params)

print("  CatBoost")
oof_catboost = get_oof_probabilities('catboost', X_train_orig, y_train_orig, catboost_best_params)

print("  LightGBM")
oof_lgbm = get_oof_probabilities('lgbm', X_train_label, y_train_label, lgbm_best_params)

print("  Logistic Regression")
oof_logreg = get_oof_probabilities('logreg', X_train_scaled, y_train_scaled, logreg_best_params)

print("\nOOF probabilities generated for all models\n")

#Find optimal threshold from OOF predictions (training data only)
optimal_threshold_rf,       oof_f2_rf       = find_optimal_threshold_oof(oof_rf,       y_train_onehot)
optimal_threshold_xgb,      oof_f2_xgb      = find_optimal_threshold_oof(oof_xgb,      y_train_onehot)
optimal_threshold_catboost, oof_f2_catboost = find_optimal_threshold_oof(oof_catboost, y_train_orig)
optimal_threshold_lgbm,     oof_f2_lgbm     = find_optimal_threshold_oof(oof_lgbm,     y_train_label)
optimal_threshold_logreg,   oof_f2_logreg   = find_optimal_threshold_oof(oof_logreg,   y_train_scaled)

print("OPTIMAL THRESHOLDS (selected on OOF training predictions)")
print(f"Random Forest:        threshold={optimal_threshold_rf:.2f}  OOF F2={oof_f2_rf:.4f}")
print(f"XGBoost:              threshold={optimal_threshold_xgb:.2f}  OOF F2={oof_f2_xgb:.4f}")
print(f"CatBoost:             threshold={optimal_threshold_catboost:.2f}  OOF F2={oof_f2_catboost:.4f}")
print(f"LightGBM:             threshold={optimal_threshold_lgbm:.2f}  OOF F2={oof_f2_lgbm:.4f}")
print(f"Logistic Regression:  threshold={optimal_threshold_logreg:.2f}  OOF F2={oof_f2_logreg:.4f}")
print("Note: OOF F2 values above are on training data and will differ from")
print("      test set F2 values reported below - that gap reflects real generalisation.\n")

Generating OOF probabilities (this runs 5-fold CV per model)
  Random Forest
  XGBoost
  CatBoost
  LightGBM
  Logistic Regression

OOF probabilities generated for all models

OPTIMAL THRESHOLDS (selected on OOF training predictions)
Random Forest:        threshold=0.38  OOF F2=0.7339
XGBoost:              threshold=0.41  OOF F2=0.7547
CatBoost:             threshold=0.34  OOF F2=0.7505
LightGBM:             threshold=0.37  OOF F2=0.7505
Logistic Regression:  threshold=0.44  OOF F2=0.7531
Note: OOF F2 values above are on training data and will differ from
      test set F2 values reported below - that gap reflects real generalisation.



## 5. Ensemble Learning

Four ensemble strategies are evaluated: soft voting (mean probabilities), weighted average (F2-score weights), hard voting (majority), and stacking (Logistic Regression meta-learner). Thresholds are optimized on OOF training data only.

In [31]:
print("ENSEMBLE LEARNING APPROACH (INCLUDING CATBOOST)")
print("This section implements multiple ensemble strategies:")
print("  1. Soft Voting Classifier (average probabilities)")
print("  2. Weighted Average (based on F2 scores)")
print("  3. Hard Voting Classifier (majority vote)")
print("  4. Stacking Ensemble (meta-learner) - NO DATA LEAKAGE")
print("  5. Comprehensive comparison with base models")

#Prepare base models for ensemble learning
print("Step 1: Base-model summary")

base_models_info = {
    'Random Forest': {
        'model':    rf_model,
        'X_train':  X_train_rf_resampled,
        'y_train':  y_train_rf_resampled,
        'X_test':   X_test_onehot,
        'y_test':   y_test_onehot,
        'optimal_threshold': None,
        'categorical_features': None
    },
    'XGBoost': {
        'model':    xgb_model,
        'X_train':  X_train_onehot,
        'y_train':  y_train_onehot,
        'X_test':   X_test_onehot,
        'y_test':   y_test_onehot,
        'optimal_threshold': None,
        'categorical_features': None
    },
    'LightGBM': {
        'model':    lgbm_model,
        'X_train':  X_train_label,
        'y_train':  y_train_label,
        'X_test':   X_test_label,
        'y_test':   y_test_label,
        'optimal_threshold': None,
        'categorical_features': None
    },
    'Logistic Regression': {
        'model':    logreg_model,
        'X_train':  X_train_lr_resampled,
        'y_train':  y_train_lr_resampled,
        'X_test':   X_test_scaled,
        'y_test':   y_test_scaled,
        'optimal_threshold': None,
        'categorical_features': None
    },
    'CatBoost': {
        'model':    catboost_model,
        'X_train':  X_train_orig,
        'y_train':  y_train_orig,
        'X_test':   X_test_orig,
        'y_test':   y_test_orig,
        'optimal_threshold': None,
        'categorical_features': cat_features_indices
    }
}

print(f"Total base models: {len(base_models_info)}")
for name in base_models_info.keys():
    print(f"  - {name}")

#Extract optimal thresholds
print("Step 2: Extract optimal thresholds")

base_models_info['Random Forest']['optimal_threshold']       = optimal_threshold_rf
base_models_info['XGBoost']['optimal_threshold']             = optimal_threshold_xgb
base_models_info['CatBoost']['optimal_threshold']            = optimal_threshold_catboost
base_models_info['LightGBM']['optimal_threshold']            = optimal_threshold_lgbm
base_models_info['Logistic Regression']['optimal_threshold'] = optimal_threshold_logreg

for name, info in base_models_info.items():
    print(f"  {name}: threshold = {info['optimal_threshold']:.2f}")

#find_optimal_threshold_oof is defined in the helper functions cell above.
#Generate base-model predictions and metrics
print("Step 3: Base-model predictions")

f2_scorer          = make_scorer(fbeta_score, beta=2)
base_predictions   = {}
base_probabilities = {}
base_metrics       = {}

for name, info in base_models_info.items():
    model     = info['model']
    X_test    = info['X_test']
    y_test    = info['y_test']
    threshold = info['optimal_threshold']

    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        base_probabilities[name] = y_proba

        y_pred = (y_proba >= threshold).astype(int)
        base_predictions[name] = y_pred

        base_metrics[name] = {
            'F2_Score':  fbeta_score(y_test, y_pred, beta=2),
            'Accuracy':  accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall':    recall_score(y_test, y_pred, zero_division=0),
            'ROC_AUC':   roc_auc_score(y_test, y_proba),
            'AP':        average_precision_score(y_test, y_proba),
            'Threshold': threshold
        }

        print(f"{name}: F2={base_metrics[name]['F2_Score']:.3f}  "
              f"Recall={base_metrics[name]['Recall']:.3f}  "
              f"Precision={base_metrics[name]['Precision']:.3f}  "
              f"ROC-AUC={base_metrics[name]['ROC_AUC']:.3f}  "
              f"threshold={threshold:.2f}")

    except Exception as e:
        print(f"{name}: Error - {str(e)}")

#Generate OOF ensemble probabilities for threshold optimization
#Derive OOF ensemble probabilities from individual model OOF arrays
#Threshold sweep uses the same leak-free approach as the base models
#Test set excluded during threshold selection
print("Step 4: Building OOF ensemble probabilities for threshold optimization")

model_names = list(base_models_info.keys())
oof_probabilities = {
    'Random Forest': oof_rf,
    'XGBoost': oof_xgb,
    'LightGBM': oof_lgbm,
    'Logistic Regression': oof_logreg,
    'CatBoost': oof_catboost,
}
oof_arrays = [oof_probabilities[name] for name in model_names]

oof_f2_scores = {
    'Random Forest': oof_f2_rf,
    'XGBoost': oof_f2_xgb,
    'LightGBM': oof_f2_lgbm,
    'Logistic Regression': oof_f2_logreg,
    'CatBoost': oof_f2_catboost,
}

f2_scores_list = [oof_f2_scores[n] for n in model_names]
oof_weights    = np.array(f2_scores_list, dtype=float) / np.sum(f2_scores_list)


#Soft voting OOF
oof_soft_proba = np.mean(oof_arrays, axis=0)

#Weighted average OOF
oof_weighted_proba = np.average(oof_arrays, axis=0, weights=oof_weights)

#Hard voting OOF - binarize each model using its own OOF threshold first
oof_hard_votes = np.column_stack([
    (oof_rf       >= optimal_threshold_rf).astype(int),
    (oof_xgb      >= optimal_threshold_xgb).astype(int),
    (oof_catboost >= optimal_threshold_catboost).astype(int),
    (oof_lgbm     >= optimal_threshold_lgbm).astype(int),
    (oof_logreg   >= optimal_threshold_logreg).astype(int),
])
oof_hard_proba = oof_hard_votes.mean(axis=1)

#Find optimal threshold for each ensemble on OOF training data only
opt_t_soft,     oof_f2_soft     = find_optimal_threshold_oof(oof_soft_proba,     y_train_onehot)
opt_t_weighted, oof_f2_weighted = find_optimal_threshold_oof(oof_weighted_proba, y_train_onehot)
opt_t_hard,     oof_f2_hard     = find_optimal_threshold_oof(oof_hard_proba,     y_train_onehot)

print("Ensemble thresholds selected on OOF training data (no leakage):")
print(f"  Soft Voting     : threshold={opt_t_soft:.2f}  OOF F2={oof_f2_soft:.3f}")
print(f"  Weighted Average: threshold={opt_t_weighted:.2f}  OOF F2={oof_f2_weighted:.3f}")
print(f"  Hard Voting     : threshold={opt_t_hard:.2f}  OOF F2={oof_f2_hard:.3f}")
print("  (Stacking threshold will be optimized after meta-learner is trained)")

#Soft Voting
print("STEP 5: Soft Voting Ensemble (Average Probabilities)")

ensemble_soft_proba = np.mean(list(base_probabilities.values()), axis=0)
ensemble_soft_pred  = (ensemble_soft_proba >= opt_t_soft).astype(int)

ensemble_soft_metrics = {
    'F2_Score':  fbeta_score(y_test_onehot, ensemble_soft_pred, beta=2),
    'Accuracy':  accuracy_score(y_test_onehot, ensemble_soft_pred),
    'Precision': precision_score(y_test_onehot, ensemble_soft_pred, zero_division=0),
    'Recall':    recall_score(y_test_onehot, ensemble_soft_pred, zero_division=0),
    'ROC_AUC':   roc_auc_score(y_test_onehot, ensemble_soft_proba),
    'AP':        average_precision_score(y_test_onehot, ensemble_soft_proba),
    'Threshold': opt_t_soft
}

print(f"Soft Voting - F2={ensemble_soft_metrics['F2_Score']:.3f}  "
      f"Recall={ensemble_soft_metrics['Recall']:.3f}  "
      f"Precision={ensemble_soft_metrics['Precision']:.3f}  "
      f"threshold={opt_t_soft:.2f}")

#Weighted Average
print("STEP 6: Weighted Average Ensemble (F2-Score Weighted)")

print("Model weights based on F2-Score:")
for name, weight in zip(model_names, oof_weights):
    print(f"  {name}: {weight:.3f}")

ensemble_weighted_proba = np.average(list(base_probabilities.values()),
                                     axis=0, weights=oof_weights)
ensemble_weighted_pred  = (ensemble_weighted_proba >= opt_t_weighted).astype(int)

ensemble_weighted_metrics = {
    'F2_Score':  fbeta_score(y_test_onehot, ensemble_weighted_pred, beta=2),
    'Accuracy':  accuracy_score(y_test_onehot, ensemble_weighted_pred),
    'Precision': precision_score(y_test_onehot, ensemble_weighted_pred, zero_division=0),
    'Recall':    recall_score(y_test_onehot, ensemble_weighted_pred, zero_division=0),
    'ROC_AUC':   roc_auc_score(y_test_onehot, ensemble_weighted_proba),
    'AP':        average_precision_score(y_test_onehot, ensemble_weighted_proba),
    'Threshold': opt_t_weighted
}

print(f"Weighted Average - F2={ensemble_weighted_metrics['F2_Score']:.3f}  "
      f"Recall={ensemble_weighted_metrics['Recall']:.3f}  "
      f"Precision={ensemble_weighted_metrics['Precision']:.3f}  "
      f"threshold={opt_t_weighted:.2f}")

#Hard Voting
print("STEP 7: Hard Voting Ensemble (Majority Vote)")

test_votes = np.column_stack([
    (base_probabilities['Random Forest']       >= optimal_threshold_rf).astype(int),
    (base_probabilities['XGBoost']             >= optimal_threshold_xgb).astype(int),
    (base_probabilities['CatBoost']            >= optimal_threshold_catboost).astype(int),
    (base_probabilities['LightGBM']            >= optimal_threshold_lgbm).astype(int),
    (base_probabilities['Logistic Regression'] >= optimal_threshold_logreg).astype(int),
])

#Apply OOF-optimized threshold to vote fraction
ensemble_hard_pred  = (test_votes.mean(axis=1) >= opt_t_hard).astype(int)
ensemble_hard_proba = ensemble_soft_proba  #Soft probabilities used for ROC-AUC

ensemble_hard_metrics = {
    'F2_Score':  fbeta_score(y_test_onehot, ensemble_hard_pred, beta=2),
    'Accuracy':  accuracy_score(y_test_onehot, ensemble_hard_pred),
    'Precision': precision_score(y_test_onehot, ensemble_hard_pred, zero_division=0),
    'Recall':    recall_score(y_test_onehot, ensemble_hard_pred, zero_division=0),
    'ROC_AUC':   roc_auc_score(y_test_onehot, ensemble_hard_proba),
    'AP':        average_precision_score(y_test_onehot, ensemble_hard_proba),
    'Threshold': opt_t_hard
}

print(f"Hard Voting - F2={ensemble_hard_metrics['F2_Score']:.3f}  "
      f"Recall={ensemble_hard_metrics['Recall']:.3f}  "
      f"Precision={ensemble_hard_metrics['Precision']:.3f}  "
      f"threshold={opt_t_hard:.2f}")
print("  Note: ROC-AUC uses averaged soft probabilities (not binary vote means)")

#Stacking with meta-learner
print("STEP 8: Stacking Ensemble (Logistic Regression as Meta-Learner)")

print("Generating out-of-fold predictions")
meta_train_features = []

#Random Forest - SMOTEENN inside each fold
print("  Random Forest OOF")
skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba_rf = np.zeros(len(X_train_onehot))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_onehot, y_train_onehot)):
    X_fold_train = X_train_onehot.iloc[train_idx]
    X_fold_val   = X_train_onehot.iloc[val_idx]
    y_fold_train = y_train_onehot.iloc[train_idx]

    smote_enn = SMOTEENN(random_state=SEED)
    X_fold_resampled, y_fold_resampled = smote_enn.fit_resample(X_fold_train, y_fold_train)

    fold_model = RandomForestClassifier(**study_rf.best_params, random_state=SEED, n_jobs=-1)
    fold_model.fit(X_fold_resampled, y_fold_resampled)
    oof_proba_rf[val_idx] = fold_model.predict_proba(X_fold_val)[:, 1]

meta_train_features.append(oof_proba_rf)
print("    done")

#XGBoost
print("  XGBoost OOF")
oof_proba_xgb = cross_val_predict(
    xgb_model, X_train_onehot, y_train_onehot,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    method='predict_proba', n_jobs=-1
)[:, 1]
meta_train_features.append(oof_proba_xgb)
print("    done")

#LightGBM
print("  LightGBM OOF")
oof_proba_lgbm = np.zeros(len(X_train_label))
for fold, (train_idx, val_idx) in enumerate(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train_label, y_train_label)):
    X_fold_train = X_train_label.iloc[train_idx]
    X_fold_val   = X_train_label.iloc[val_idx]
    y_fold_train = y_train_label.iloc[train_idx]

    fold_model = LGBMClassifier(**lgbm_model.get_params())
    fold_model.fit(X_fold_train, y_fold_train, categorical_feature='auto')
    oof_proba_lgbm[val_idx] = fold_model.predict_proba(X_fold_val)[:, 1]

meta_train_features.append(oof_proba_lgbm)
print("    done")

#Logistic Regression - SMOTEENN inside each fold
print("  Logistic Regression OOF")
oof_proba_lr        = np.zeros(len(X_train_scaled))
logreg_params_fixed = logreg_params_final.copy()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train_scaled)):
    X_fold_train = X_train_scaled.iloc[train_idx]
    X_fold_val   = X_train_scaled.iloc[val_idx]
    y_fold_train = y_train_scaled.iloc[train_idx]

    smote_enn = SMOTEENN(random_state=SEED)
    X_fold_resampled, y_fold_resampled = smote_enn.fit_resample(X_fold_train, y_fold_train)

    fold_model = LogisticRegression(**logreg_params_fixed)
    fold_model.fit(X_fold_resampled, y_fold_resampled)
    oof_proba_lr[val_idx] = fold_model.predict_proba(X_fold_val)[:, 1]

meta_train_features.append(oof_proba_lr)
print("    done")

#CatBoost
print("  CatBoost OOF")
oof_proba_catboost = np.zeros(len(X_train_orig))
for fold, (train_idx, val_idx) in enumerate(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train_orig, y_train_orig)):
    X_fold_train = X_train_orig.iloc[train_idx]
    X_fold_val   = X_train_orig.iloc[val_idx]
    y_fold_train = y_train_orig.iloc[train_idx]

    fold_model = CatBoostClassifier(**catboost_model.get_params())
    fold_model.fit(X_fold_train, y_fold_train,
                   cat_features=cat_features_indices, verbose=False)
    oof_proba_catboost[val_idx] = fold_model.predict_proba(X_fold_val)[:, 1]

meta_train_features.append(oof_proba_catboost)
print("    done")

#Verify dimensions
stacking_model_names = [
    'Random Forest',
    'XGBoost',
    'LightGBM',
    'Logistic Regression',
    'CatBoost',
]

print(f"\nVerifying meta-feature dimensions:")
for name, features in zip(stacking_model_names, meta_train_features):
    print(f"  {name}: {len(features)} samples")

meta_train_X = pd.DataFrame(
    np.column_stack(meta_train_features),
    columns=stacking_model_names
)
meta_train_y = y_train_onehot
meta_test_X  = pd.DataFrame(
    np.column_stack([base_probabilities[name] for name in stacking_model_names]),
    columns=stacking_model_names
)

print(f"\nMeta-features: train={meta_train_X.shape}  test={meta_test_X.shape}")

print("\nTraining meta-learner (Logistic Regression)")
meta_learner = LogisticRegression(random_state=SEED, max_iter=1000)
meta_learner.fit(meta_train_X, meta_train_y)
print("Meta-learner trained")

print("\nMeta-learner weights per base model:")
for name, weight in zip(stacking_model_names, meta_learner.coef_[0]):
    print(f"  {name}: {weight:.3f}")

#Optimize stacking threshold on OOF training predictions
oof_stacking_proba      = meta_learner.predict_proba(meta_train_X)[:, 1]
opt_t_stacking, oof_f2_stacking = find_optimal_threshold_oof(oof_stacking_proba, y_train_onehot)
print(f"\nStacking threshold selected on OOF training data: "
      f"threshold={opt_t_stacking:.2f}  OOF F2={oof_f2_stacking:.3f}")

#Apply optimized threshold to test set
ensemble_stacking_proba = meta_learner.predict_proba(meta_test_X)[:, 1]
ensemble_stacking_pred  = (ensemble_stacking_proba >= opt_t_stacking).astype(int)

ensemble_stacking_metrics = {
    'F2_Score':  fbeta_score(y_test_onehot, ensemble_stacking_pred, beta=2),
    'Accuracy':  accuracy_score(y_test_onehot, ensemble_stacking_pred),
    'Precision': precision_score(y_test_onehot, ensemble_stacking_pred, zero_division=0),
    'Recall':    recall_score(y_test_onehot, ensemble_stacking_pred, zero_division=0),
    'ROC_AUC':   roc_auc_score(y_test_onehot, ensemble_stacking_proba),
    'AP':        average_precision_score(y_test_onehot, ensemble_stacking_proba),
    'Threshold': opt_t_stacking
}

print(f"Stacking - F2={ensemble_stacking_metrics['F2_Score']:.3f}  "
      f"Recall={ensemble_stacking_metrics['Recall']:.3f}  "
      f"Precision={ensemble_stacking_metrics['Precision']:.3f}  "
      f"threshold={opt_t_stacking:.2f}")

#Comprehensive comparison table
print("COMPREHENSIVE COMPARISON: BASE MODELS VS ENSEMBLE METHODS")

comparison_data = []

for name, metrics in base_metrics.items():
    comparison_data.append({'Model': name, 'Type': 'Base Model', **metrics})

ensemble_methods = {
    'Soft Voting':      ensemble_soft_metrics,
    'Weighted Average': ensemble_weighted_metrics,
    'Hard Voting':      ensemble_hard_metrics,
    'Stacking':         ensemble_stacking_metrics,
}

for name, metrics in ensemble_methods.items():
    comparison_data.append({'Model': name, 'Type': 'Ensemble', **metrics})

comparison_df = pd.DataFrame(comparison_data).sort_values('F2_Score', ascending=False).reset_index(drop=True)

print("\n")
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

best_model      = comparison_df.iloc[0]
best_base_model = comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]
best_ensemble   = comparison_df[comparison_df['Type'] == 'Ensemble'].iloc[0]
improvement     = ((best_ensemble['F2_Score'] - best_base_model['F2_Score'])
                   / best_base_model['F2_Score']) * 100

print(f"BEST PERFORMING MODEL: {best_model['Model']} ({best_model['Type']})")
print(f"  F2-Score:  {best_model['F2_Score']:.3f}")
print(f"  Recall:    {best_model['Recall']:.3f}")
print(f"  Precision: {best_model['Precision']:.3f}")
print(f"  ROC-AUC:   {best_model['ROC_AUC']:.3f}")
print(f"  AP:    {best_model['AP']:.3f}")
print(f"  Threshold: {best_model['Threshold']:.2f}")

print(f"Best Base Model: {best_base_model['Model']} (F2={best_base_model['F2_Score']:.3f})")
print(f"Best Ensemble:   {best_ensemble['Model']} (F2={best_ensemble['F2_Score']:.3f})")
print(f"Improvement:     {improvement:+.2f}%")

all_positive_pred = np.ones_like(y_test_onehot)
all_positive_baseline = {
    'F2_Score':  fbeta_score(y_test_onehot, all_positive_pred, beta=2),
    'Accuracy':  accuracy_score(y_test_onehot, all_positive_pred),
    'Precision': precision_score(y_test_onehot, all_positive_pred, zero_division=0),
    'Recall':    recall_score(y_test_onehot, all_positive_pred, zero_division=0),
    'Positive_Prevalence': float(np.mean(y_test_onehot)),
    'Best_Model_F2_Lift': float(best_model['F2_Score'] - fbeta_score(y_test_onehot, all_positive_pred, beta=2)),
}

print("ALL-POSITIVE BASELINE (predict every customer as churn)")
print(f"  Positive prevalence / baseline precision: {all_positive_baseline['Positive_Prevalence']:.3f}")
print(f"  Baseline F2:                         {all_positive_baseline['F2_Score']:.3f}")
print(f"  Best model F2 lift over baseline:    {all_positive_baseline['Best_Model_F2_Lift']:+.3f}")

ensemble_results = {
    'comparison_df':           comparison_df,
    'base_probabilities':      base_probabilities,
    'ensemble_soft_proba':     ensemble_soft_proba,
    'ensemble_weighted_proba': ensemble_weighted_proba,
    'ensemble_hard_proba':     ensemble_hard_proba,
    'ensemble_stacking_proba': ensemble_stacking_proba,
    'ensemble_soft_pred':      ensemble_soft_pred,
    'ensemble_weighted_pred':  ensemble_weighted_pred,
    'ensemble_hard_pred':      ensemble_hard_pred,
    'ensemble_stacking_pred':  ensemble_stacking_pred,
    'y_test':                  y_test_onehot,
    'ensemble_thresholds': {
        'Soft Voting':      opt_t_soft,
        'Weighted Average': opt_t_weighted,
        'Hard Voting':      opt_t_hard,
        'Stacking':         opt_t_stacking,
    },
    'all_positive_baseline': all_positive_baseline,
}

print("\nEnsemble learning completed successfully!")

ENSEMBLE LEARNING APPROACH (INCLUDING CATBOOST)
This section implements multiple ensemble strategies:
  1. Soft Voting Classifier (average probabilities)
  2. Weighted Average (based on F2 scores)
  3. Hard Voting Classifier (majority vote)
  4. Stacking Ensemble (meta-learner) - NO DATA LEAKAGE
  5. Comprehensive comparison with base models
Step 1: Base-model summary
Total base models: 5
  - Random Forest
  - XGBoost
  - LightGBM
  - Logistic Regression
  - CatBoost
Step 2: Extract optimal thresholds
  Random Forest: threshold = 0.38
  XGBoost: threshold = 0.41
  LightGBM: threshold = 0.37
  Logistic Regression: threshold = 0.44
  CatBoost: threshold = 0.34
Step 3: Base-model predictions
Random Forest: F2=0.730  Recall=0.933  Precision=0.391  ROC-AUC=0.811  threshold=0.38
XGBoost: F2=0.743  Recall=0.882  Precision=0.455  ROC-AUC=0.841  threshold=0.41
LightGBM: F2=0.738  Recall=0.888  Precision=0.441  ROC-AUC=0.839  threshold=0.37
Logistic Regression: F2=0.742  Recall=0.893  Precision=

/home/maryam/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/maryam/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain feature

    done
  LightGBM OOF
    done
  Logistic Regression OOF
    done
  CatBoost OOF
    done

Verifying meta-feature dimensions:
  Random Forest: 5625 samples
  XGBoost: 5625 samples
  LightGBM: 5625 samples
  Logistic Regression: 5625 samples
  CatBoost: 5625 samples

Meta-features: train=(5625, 5)  test=(1407, 5)

Training meta-learner (Logistic Regression)
Meta-learner trained

Meta-learner weights per base model:
  Random Forest: 0.357
  XGBoost: 1.830
  LightGBM: 1.323
  Logistic Regression: -0.256
  CatBoost: 2.453

Stacking threshold selected on OOF training data: threshold=0.14  OOF F2=0.752
Stacking - F2=0.739  Recall=0.890  Precision=0.440  threshold=0.14
COMPREHENSIVE COMPARISON: BASE MODELS VS ENSEMBLE METHODS


              Model       Type  F2_Score  Accuracy  Precision  Recall  ROC_AUC    AP  Threshold
            XGBoost Base Model     0.743     0.687      0.455   0.882    0.841 0.650      0.410
Logistic Regression Base Model     0.742     0.673      0.443   0.893    0.

## 6. Model Explainability (SHAP)

SHAP values are computed for all five base models and for ensemble strategies. Tree models use `TreeExplainer`; Logistic Regression uses `LinearExplainer` with a 500-row background sample.

In [32]:
import re

#Helper function: Print Feature Importance Rankings (Console Only)

def print_feature_importance(shap_values, feature_names, model_name, top_n=10):
    """
    Print top N feature importance rankings in console only.
    Reports mean and standard deviation of |SHAP| across samples.
    """
    abs_shap      = np.abs(shap_values)
    mean_abs_shap = abs_shap.mean(axis=0)
    std_abs_shap  = abs_shap.std(axis=0)

    importance_df = pd.DataFrame({
        'Feature':     feature_names,
        'Mean_|SHAP|': mean_abs_shap,
        'Std_|SHAP|':  std_abs_shap,
    }).sort_values('Mean_|SHAP|', ascending=False).reset_index(drop=True)
    importance_df['Rank'] = range(1, len(importance_df) + 1)

    print(f"\n{'='*95}")
    print(f"{model_name} - Top {top_n} Most Important Features")
    print(f"{'='*95}")
    print(f"{'Rank':<6} {'Feature':<50} {'Mean |SHAP|':<15} {'Std |SHAP|':<15}")
    print(f"{'-'*95}")
    for _, row in importance_df.head(top_n).iterrows():
        print(f"{int(row['Rank']):<6} {row['Feature']:<50} "
              f"{row['Mean_|SHAP|']:<15.3f} {row['Std_|SHAP|']:<15.3f}")
    print(f"{'-'*95}")
    print(f"Total features analysed: {len(importance_df)}")

    return importance_df


#Helper function: Aggregate one-hot SHAP back to original feature names

def aggregate_onehot_shap(shap_vals, feature_names, categorical_cols):
    """
    Rolls one-hot SHAP values back to original categorical feature names.
    Works with pd.get_dummies naming convention: 'ServiceArea_OHIMED330' -> 'ServiceArea'
    Aggregates |SHAP| per-sample first, then computes mean and std across samples.
    """
    #Map each one-hot column to its base (original) feature name
    base_names = []
    for fname in feature_names:
        base = fname
        for cat in categorical_cols:
            if fname.startswith(cat + '_') or fname == cat:
                base = cat
                break
        base_names.append(base)

    abs_shap     = np.abs(shap_vals)
    unique_bases = list(dict.fromkeys(base_names))  #Preserve first-seen order

    #Sum |SHAP| per sample across one-hot columns belonging to the same base feature
    agg_per_sample = np.zeros((abs_shap.shape[0], len(unique_bases)))
    for i, base in enumerate(unique_bases):
        mask = np.array([b == base for b in base_names])
        agg_per_sample[:, i] = abs_shap[:, mask].sum(axis=1)

    result = pd.DataFrame({
        'Feature':     unique_bases,
        'Mean_|SHAP|': agg_per_sample.mean(axis=0),
        'Std_|SHAP|':  agg_per_sample.std(axis=0),
    }).sort_values('Mean_|SHAP|', ascending=False).reset_index(drop=True)
    result['Rank'] = range(1, len(result) + 1)

    return result


def print_aggregated_importance(importance_df, model_name, top_n=10):
    print(f"\n{'='*95}")
    print(f"{model_name} - Top {top_n} Most Important Features (Aggregated)")
    print(f"{'='*95}")
    print(f"{'Rank':<6} {'Feature':<50} {'Mean |SHAP|':<15} {'Std |SHAP|':<15}")
    print(f"{'-'*95}")
    for _, row in importance_df.head(top_n).iterrows():
        print(f"{int(row['Rank']):<6} {row['Feature']:<50} "
              f"{row['Mean_|SHAP|']:<15.3f} {row['Std_|SHAP|']:<15.3f}")
    print(f"{'-'*95}")
    print(f"Total features analysed: {len(importance_df)}")


#SHAP analysis for base models

print("Part 1: Base-model SHAP analysis")

base_shap_values        = {}
base_explainers         = {}
base_feature_importance = {}
base_feature_importance_aggregated = {}

#1 Random Forest SHAP
print("1. Random Forest")

try:
    explainer_rf   = shap.TreeExplainer(rf_model)
    shap_values_rf = explainer_rf.shap_values(X_train_onehot)

    if isinstance(shap_values_rf, list):
        shap_values_rf = shap_values_rf[1]

    base_shap_values['Random Forest'] = shap_values_rf
    base_explainers['Random Forest']  = explainer_rf

    feature_names = (X_train_onehot.columns.tolist()
                     if hasattr(X_train_onehot, 'columns')
                     else [f'Feature_{i}' for i in range(X_train_onehot.shape[1])])

    print(f"SHAP values computed  shape={shap_values_rf.shape}")
    base_feature_importance['Random Forest'] = print_feature_importance(
        shap_values_rf, feature_names, "Random Forest", top_n=10
    )
    base_feature_importance_aggregated['Random Forest'] = aggregate_onehot_shap(
        shap_values_rf, feature_names, categorical_cols_onehot
    )
    print_aggregated_importance(
        base_feature_importance_aggregated['Random Forest'],
        "Random Forest", top_n=10
    )

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_rf, X_train_onehot,
                      plot_type="bar", max_display=10, show=False)
    plt.title("Random Forest - Top 10 Features (SHAP)")
    plt.tight_layout()
    plt.savefig('shap_rf_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Plot saved: shap_rf_summary.png")

except Exception as e:
    print(f"Random Forest SHAP failed: {str(e)}")

#2 XGBoost SHAP
print("2. XGBoost")

try:
    explainer_xgb   = shap.TreeExplainer(xgb_model)
    shap_values_xgb = explainer_xgb.shap_values(X_train_onehot)

    base_shap_values['XGBoost'] = shap_values_xgb
    base_explainers['XGBoost']  = explainer_xgb

    print(f"SHAP values computed  shape={shap_values_xgb.shape}")
    base_feature_importance['XGBoost'] = print_feature_importance(
        shap_values_xgb, feature_names, "XGBoost", top_n=10
    )
    base_feature_importance_aggregated['XGBoost'] = aggregate_onehot_shap(
        shap_values_xgb, feature_names, categorical_cols_onehot
    )
    print_aggregated_importance(
        base_feature_importance_aggregated['XGBoost'],
        "XGBoost", top_n=10
    )

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_xgb, X_train_onehot,
                      plot_type="bar", max_display=10, show=False)
    plt.title("XGBoost - Top 10 Features (SHAP)")
    plt.tight_layout()
    plt.savefig('shap_xgb_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Plot saved: shap_xgb_summary.png")

except Exception as e:
    print(f"XGBoost SHAP failed: {str(e)}")

#3 LightGBM SHAP
print("3. LightGBM")

try:
    explainer_lgbm   = shap.TreeExplainer(lgbm_model)
    shap_values_lgbm = explainer_lgbm.shap_values(X_train_label)

    if isinstance(shap_values_lgbm, list):
        shap_values_lgbm = shap_values_lgbm[1]

    base_shap_values['LightGBM'] = shap_values_lgbm
    base_explainers['LightGBM']  = explainer_lgbm

    feature_names_lgbm = (X_train_label.columns.tolist()
                          if hasattr(X_train_label, 'columns')
                          else [f'Feature_{i}' for i in range(X_train_label.shape[1])])

    print(f"SHAP values computed  shape={shap_values_lgbm.shape}")
    base_feature_importance['LightGBM'] = print_feature_importance(
        shap_values_lgbm, feature_names_lgbm, "LightGBM", top_n=10
    )

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_lgbm, X_train_label,
                      plot_type="bar", max_display=10, show=False)
    plt.title("LightGBM - Top 10 Features (SHAP)")
    plt.tight_layout()
    plt.savefig('shap_lgbm_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Plot saved: shap_lgbm_summary.png")

except Exception as e:
    print(f"LightGBM SHAP failed: {str(e)}")

#4 CatBoost SHAP
print("4. CatBoost")

try:
    explainer_catboost   = shap.TreeExplainer(catboost_model)
    shap_values_catboost = explainer_catboost.shap_values(X_train_orig)

    base_shap_values['CatBoost'] = shap_values_catboost
    base_explainers['CatBoost']  = explainer_catboost

    feature_names_catboost = (X_train_orig.columns.tolist()
                              if hasattr(X_train_orig, 'columns')
                              else [f'Feature_{i}' for i in range(X_train_orig.shape[1])])

    print(f"SHAP values computed  shape={shap_values_catboost.shape}")
    base_feature_importance['CatBoost'] = print_feature_importance(
        shap_values_catboost, feature_names_catboost, "CatBoost", top_n=10
    )

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_catboost, X_train_orig,
                      plot_type="bar", max_display=10, show=False)
    plt.title("CatBoost - Top 10 Features (SHAP)")
    plt.tight_layout()
    plt.savefig('shap_catboost_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Plot saved: shap_catboost_summary.png")

except Exception as e:
    print(f"CatBoost SHAP failed: {str(e)}")

#5 Logistic Regression SHAP
#Uses full X_train_scaled as background for accurate SHAP values
#Aggregates one-hot columns back to original feature names
print("5. Logistic Regression")

try:
    explainer_logreg   = shap.LinearExplainer(logreg_model, X_train_scaled)
    shap_values_logreg = explainer_logreg.shap_values(X_train_scaled)

    base_shap_values['Logistic Regression'] = shap_values_logreg
    base_explainers['Logistic Regression']  = explainer_logreg

    feature_names_scaled = (X_train_scaled.columns.tolist()
                            if hasattr(X_train_scaled, 'columns')
                            else [f'Feature_{i}' for i in range(X_train_scaled.shape[1])])

    print(f"SHAP values computed  shape={shap_values_logreg.shape}")

    aggregated_importance = aggregate_onehot_shap(
        shap_values_logreg, feature_names_scaled, categorical_cols_scaled
    )
    base_feature_importance['Logistic Regression'] = aggregated_importance
    base_feature_importance_aggregated['Logistic Regression'] = aggregated_importance
    print_aggregated_importance(aggregated_importance, "Logistic Regression", top_n=10)

    plt.figure(figsize=(10, 6))
    plot_data = aggregated_importance.head(10).sort_values('Mean_|SHAP|', ascending=True)
    plt.barh(plot_data['Feature'], plot_data['Mean_|SHAP|'],
             xerr=plot_data['Std_|SHAP|'], color='steelblue',
             error_kw={'ecolor': 'black', 'capsize': 3})
    plt.xlabel('Mean |SHAP Value|')
    plt.title("Logistic Regression - Top 10 Features (SHAP, Aggregated)")
    plt.tight_layout()
    plt.savefig('shap_logreg_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Plot saved: shap_logreg_summary.png")

except Exception as e:
    print(f"Logistic Regression SHAP failed: {str(e)}")

print("SHAP ANALYSIS COMPLETE")
print("Files saved: shap_rf_summary.png, shap_xgb_summary.png,")
print("             shap_lgbm_summary.png, shap_catboost_summary.png,")
print("             shap_logreg_summary.png")

Part 1: Base-model SHAP analysis
1. Random Forest
SHAP values computed  shape=(5625, 35)

Random Forest - Top 10 Most Important Features
Rank   Feature                                            Mean |SHAP|     Std |SHAP|     
-----------------------------------------------------------------------------------------------
1      tenure                                             0.065           0.018          
2      InternetService_Fiber optic                        0.038           0.004          
3      Contract_Two year                                  0.035           0.026          
4      StreamingTV_No internet service                    0.019           0.015          
5      TotalCharges_log                                   0.019           0.009          
6      TechSupport_No internet service                    0.018           0.015          
7      DeviceProtection_No internet service               0.018           0.015          
8      OnlineSecurity_No internet service      

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray


Plot saved: shap_lgbm_summary.png
4. CatBoost
SHAP values computed  shape=(5625, 24)

CatBoost - Top 10 Most Important Features
Rank   Feature                                            Mean |SHAP|     Std |SHAP|     
-----------------------------------------------------------------------------------------------
1      Contract                                           0.717           0.216          
2      tenure                                             0.349           0.209          
3      OnlineSecurity                                     0.282           0.014          
4      InternetService                                    0.225           0.024          
5      TechSupport                                        0.186           0.019          
6      PaymentMethod                                      0.172           0.051          
7      MonthlyCharges_log                                 0.108           0.058          
8      MonthlyCharges                                   

In [33]:
#Ensemble-strategy SHAP on the full training data

print("Part 2: Ensemble-strategy SHAP analysis (full data)")

ensemble_shap_values      = {}
ensemble_model_importances = {}
ensemble_model_shap_std    = {}  #Store standard deviation per method

#Generate training predictions from all base models.
print(f"Generating base model predictions on all {len(X_train_onehot)} rows")

train_probabilities = {
    'Random Forest':      rf_model.predict_proba(X_train_onehot)[:, 1],
    'XGBoost':            xgb_model.predict_proba(X_train_onehot)[:, 1],
    'LightGBM':           lgbm_model.predict_proba(X_train_label)[:, 1],
    'CatBoost':           catboost_model.predict_proba(X_train_orig)[:, 1],
    'Logistic Regression': logreg_model.predict_proba(X_train_scaled)[:, 1]
}

meta_X_train = pd.DataFrame(train_probabilities)

print("Summarizing background data using K-Means (speed optimization)")
meta_background  = shap.kmeans(meta_X_train, 50)
meta_full_sample = meta_X_train

#Soft-voting SHAP
print("\n2.1 Mean Ensemble - Computing SHAP on full data")
try:
    def mean_ensemble_predict(X):
        return np.mean(X, axis=1)

    explainer_mean   = shap.KernelExplainer(mean_ensemble_predict, meta_background)
    shap_values_mean = explainer_mean.shap_values(meta_full_sample)
    ensemble_shap_values["Mean Ensemble"] = shap_values_mean

    mean_abs_shap  = np.abs(shap_values_mean).mean(axis=0)
    std_abs_shap   = np.abs(shap_values_mean).std(axis=0)
    total          = mean_abs_shap.sum()
    ensemble_model_importances["Mean Ensemble"] = dict(zip(meta_X_train.columns, mean_abs_shap / total))
    ensemble_model_shap_std["Mean Ensemble"]    = dict(zip(meta_X_train.columns, std_abs_shap / total))
    print("Mean Ensemble complete")
except Exception as e:
    print(f"Mean Ensemble failed: {e}")

#Weighted-average SHAP
print("\n2.2 Weighted Ensemble - Computing SHAP on full data")
try:
    model_names = list(train_probabilities.keys())
    f2_scores = [oof_f2_scores[name] for name in model_names]
    weights     = np.array(f2_scores) / np.sum(f2_scores)

    def weighted_ensemble_predict(X):
        return np.dot(X, weights)

    explainer_weighted   = shap.KernelExplainer(weighted_ensemble_predict, meta_background)
    shap_values_weighted = explainer_weighted.shap_values(meta_full_sample)
    ensemble_shap_values["Weighted Ensemble"] = shap_values_weighted

    mean_abs_shap = np.abs(shap_values_weighted).mean(axis=0)
    std_abs_shap  = np.abs(shap_values_weighted).std(axis=0)
    total         = mean_abs_shap.sum()
    ensemble_model_importances["Weighted Ensemble"] = dict(zip(meta_X_train.columns, mean_abs_shap / total))
    ensemble_model_shap_std["Weighted Ensemble"]    = dict(zip(meta_X_train.columns, std_abs_shap / total))
    print("Weighted Ensemble complete")
except Exception as e:
    print(f"Weighted Ensemble failed: {e}")

#3 Hard Voting SHAP
print("\n2.3 Hard Voting - Computing SHAP (Empirical Importance)")
try:
    def hard_voting_predict(X):
        votes = (X >= 0.5).astype(int)
        return (np.sum(votes, axis=1) >= (X.shape[1] / 2)).astype(int)

    explainer_hard   = shap.KernelExplainer(hard_voting_predict, meta_background)
    shap_values_hard = explainer_hard.shap_values(meta_full_sample)

    mean_abs_shap     = np.abs(shap_values_hard).mean(axis=0)
    std_abs_shap      = np.abs(shap_values_hard).std(axis=0)
    total_importance  = mean_abs_shap.sum()

    if total_importance > 0:
        norm_mean = mean_abs_shap / total_importance
        norm_std  = std_abs_shap  / total_importance
    else:
        norm_mean = np.array([1 / len(model_names)] * len(model_names))
        norm_std  = np.zeros(len(model_names))

    ensemble_model_importances["Hard Voting"] = dict(zip(meta_X_train.columns, norm_mean))
    ensemble_model_shap_std["Hard Voting"]    = dict(zip(meta_X_train.columns, norm_std))
    print("Hard Voting complete")
except Exception as e:
    print(f"Hard Voting failed: {e}")

#4 STACKING SHAP
print("\n2.4 Stacking - Computing SHAP using LinearExplainer")
try:
    meta_features = meta_train_X if 'meta_train_X' in locals() else meta_X_train
    stacking_names_for_shap = (
        list(meta_features.columns)
        if hasattr(meta_features, 'columns')
        else stacking_model_names
    )

    explainer_stacking   = shap.LinearExplainer(meta_learner, meta_features)
    shap_values_stacking = explainer_stacking.shap_values(meta_features)

    if isinstance(shap_values_stacking, list):
        shap_values_stacking = shap_values_stacking[1]

    mean_abs_shap = np.abs(shap_values_stacking).mean(axis=0)
    std_abs_shap  = np.abs(shap_values_stacking).std(axis=0)
    total         = mean_abs_shap.sum()
    ensemble_model_importances["Stacking"] = dict(zip(stacking_names_for_shap, mean_abs_shap / total))
    ensemble_model_shap_std["Stacking"]    = dict(zip(stacking_names_for_shap, std_abs_shap  / total))
    print("Stacking complete")
except Exception as e:
    print(f"Stacking failed: {e}")

#Ensemble contribution summary
print("Part 3: Ensemble contribution summary (mean ± std)")

if ensemble_model_importances:
    #Build mean and std dataframes
    mean_data, std_data = [], []

    for method, importances in ensemble_model_importances.items():
        for model, val in importances.items():
            mean_data.append({'Ensemble Method': method, 'Base Model': model, 'Mean': val})

    for method, stds in ensemble_model_shap_std.items():
        for model, val in stds.items():
            std_data.append({'Ensemble Method': method, 'Base Model': model, 'Std': val})

    mean_df = pd.DataFrame(mean_data)
    std_df  = pd.DataFrame(std_data)

    merged_df = mean_df.merge(std_df, on=['Ensemble Method', 'Base Model'])

    #Create combined "Mean ± Std" string column
    merged_df['Importance (Mean ± Std)'] = merged_df.apply(
        lambda row: f"{row['Mean']:.3f} ± {row['Std']:.3f}", axis=1
    )

    #Pivot on the combined string column for display
    pivot_display = merged_df.pivot(
        index='Base Model',
        columns='Ensemble Method',
        values='Importance (Mean ± Std)'
    )

    #Also keep numeric pivot for any downstream use
    pivot_mean = merged_df.pivot(index='Base Model', columns='Ensemble Method', values='Mean')
    pivot_std  = merged_df.pivot(index='Base Model', columns='Ensemble Method', values='Std')

    print("\nFinal Model Importance Across All Methods (Full Training Data):")
    print(pivot_display.to_string())

    #Highlight most important model per ensemble method
    print("\nMost Important Base Model per Ensemble Method:")
    for method in pivot_mean.columns:
        top_model = pivot_mean[method].idxmax()
        top_mean  = pivot_mean.loc[top_model, method]
        top_std   = pivot_std.loc[top_model, method]
        print(f"  {method:20s}: {top_model:22s}  {top_mean:.3f} ± {top_std:.3f}")


Part 2: Ensemble-strategy SHAP analysis (full data)
Generating base model predictions on all 5625 rows
Summarizing background data using K-Means (speed optimization)

2.1 Mean Ensemble - Computing SHAP on full data


100%|██████████| 5625/5625 [01:09<00:00, 80.93it/s]


Mean Ensemble complete

2.2 Weighted Ensemble - Computing SHAP on full data


100%|██████████| 5625/5625 [01:08<00:00, 82.22it/s]


Weighted Ensemble complete

2.3 Hard Voting - Computing SHAP (Empirical Importance)


100%|██████████| 5625/5625 [01:09<00:00, 81.48it/s]

Hard Voting complete

2.4 Stacking - Computing SHAP using LinearExplainer
Stacking complete
Part 3: Ensemble contribution summary (mean ± std)

Final Model Importance Across All Methods (Full Training Data):
Ensemble Method        Hard Voting  Mean Ensemble       Stacking Weighted Ensemble
Base Model                                                                        
CatBoost             0.216 ± 0.055  0.200 ± 0.100  0.408 ± 0.202     0.200 ± 0.100
LightGBM             0.211 ± 0.056  0.190 ± 0.096  0.208 ± 0.105     0.191 ± 0.097
Logistic Regression  0.188 ± 0.056  0.263 ± 0.110  0.056 ± 0.023     0.264 ± 0.110
Random Forest        0.169 ± 0.062  0.158 ± 0.097  0.046 ± 0.028     0.155 ± 0.095
XGBoost              0.215 ± 0.057  0.188 ± 0.092  0.283 ± 0.141     0.189 ± 0.093

Most Important Base Model per Ensemble Method:
  Hard Voting         : CatBoost                0.216 ± 0.055
  Mean Ensemble       : Logistic Regression     0.263 ± 0.110
  Stacking            : CatBoost       

In [34]:
y_true_xgb = base_models_info['XGBoost']['y_test']
y_pred_xgb = base_predictions['XGBoost']

cm_xgb = confusion_matrix(y_true_xgb, y_pred_xgb)
print(cm_xgb)

[[637 396]
 [ 44 330]]


In [35]:
print("\nGenerating Precision-Recall Curve (5 models, sorted legend)")

models_for_pr = [
    "LightGBM",
    "XGBoost",
    "CatBoost",
    "Logistic Regression",
    "Random Forest"
]

base_proba = {
    "LightGBM": base_probabilities["LightGBM"],
    "XGBoost": base_probabilities["XGBoost"],
    "CatBoost": base_probabilities["CatBoost"],
    "Logistic Regression": base_probabilities["Logistic Regression"],
    "Random Forest": base_probabilities["Random Forest"]
}

#Compute AP first
curve_metrics = []
for model_name in models_for_pr:
    y_scores = base_proba[model_name]
    precision, recall, _ = precision_recall_curve(y_test_onehot, y_scores)
    ap = average_precision_score(y_test_onehot, y_scores)
    curve_metrics.append((model_name, precision, recall, ap))

#Sort by AP (descending)
curve_metrics.sort(key=lambda x: x[3], reverse=True)

#Plot
plt.figure(figsize=(9, 6))
for model_name, precision, recall, ap in curve_metrics:
    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f"{model_name} (AP={ap:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve (Base Models)", fontweight="bold")
plt.legend(loc="upper right", frameon=True, fancybox=False, edgecolor="#cccccc")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pr_curve_5_models_sorted.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: pr_curve_5_models_sorted.png")


Generating Precision-Recall Curve (5 models, sorted legend)
Saved: pr_curve_5_models_sorted.png


In [36]:
print("\nGenerating ROC Curve (5 models, sorted legend)")
from sklearn.metrics import roc_curve, roc_auc_score

models_for_roc = [
    "LightGBM",
    "XGBoost",
    "CatBoost",
    "Logistic Regression",
    "Random Forest"
]

roc_proba = {
    "LightGBM": base_probabilities["LightGBM"],
    "XGBoost": base_probabilities["XGBoost"],
    "CatBoost": base_probabilities["CatBoost"],
    "Logistic Regression": base_probabilities["Logistic Regression"],
    "Random Forest": base_probabilities["Random Forest"]
}

#Compute metrics first
curve_metrics = []
for model_name in models_for_roc:
    y_scores = roc_proba[model_name]
    fpr, tpr, _ = roc_curve(y_test_onehot, y_scores)
    auc_score = roc_auc_score(y_test_onehot, y_scores)
    curve_metrics.append((model_name, fpr, tpr, auc_score))

#Sort by ROC-AUC (descending)
curve_metrics.sort(key=lambda x: x[3], reverse=True)

#Plot
plt.figure(figsize=(9, 6))
for model_name, fpr, tpr, auc_score in curve_metrics:
    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{model_name} (ROC-AUC={auc_score:.3f})"
    )

#Random baseline
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Base Models)", fontweight='bold')
plt.legend(loc="lower right", frameon=True, fancybox=False, edgecolor="#cccccc")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("roc_curve_5_models_sorted.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: roc_curve_5_models_sorted.png")


Generating ROC Curve (5 models, sorted legend)
Saved: roc_curve_5_models_sorted.png
